# 02_pipeline — thin orchestration template

Build, check, publish, and record evidence for governed Fabric data pipelines.

This notebook is intentionally thin: users edit source definitions, guardrail presets, the DIY transformation section, and target definitions. Reusable metadata, guardrail, catalogue, lineage, and runtime logging logic lives in `fabricops_kit` functions.

Flow:

1. Run `00_env_config`
2. Import required functions
3. Select data agreement and register notebook
4. Define and read many source datasets into individual DataFrames
5. Profile each source DataFrame
6. Check each source DataFrame against schema guardrails using its configured preset
7. Check each source DataFrame against data drift guardrails using its configured preset
8. Check each source DataFrame against DQ guardrails using its configured preset
9. Enrich each source profile with DQ result columns and write to `METADATA_DATA_CATALOGUE`
10. User-defined transformation section
11. Define target DataFrames and add audit columns
12. Check each target DataFrame against schema guardrails using its configured preset
13. Check each target DataFrame against data drift guardrails using its configured preset
14. Check each target DataFrame against DQ guardrails using its configured preset
15. Enrich each target profile with DQ result columns and write to `METADATA_DATA_CATALOGUE`
16. Write target tables
17. Capture many-to-many lineage tied to the notebook registry
18. Write runtime summary evidence to `METADATA_PIPELINE_RUNS`


## 1. Run `00_env_config`

Load the shared FabricOps environment, path configuration, sample metadata, and metadata lakehouse routing.

In [ ]:
%run 00_env_config


## 2. Import required functions

The notebook imports high-level orchestration helpers. Implementation detail stays inside `fabricops_kit`.

In [ ]:
from datetime import datetime, timezone

from pyspark.sql import functions as F

from fabricops_kit import (
    add_runtime_audit_columns,
    get_selected_agreement,
    profile_pipeline_datasets,
    read_pipeline_sources,
    run_data_drift_guardrails,
    run_dq_guardrails,
    run_schema_guardrails,
    stop_if_failed,
    widget_select_agreement,
    write_catalogue_evidence,
    write_pipeline_lineage,
    write_pipeline_run_summary,
    write_pipeline_targets,
)


## 3. Select data agreement and register notebook

Select the agreement that this pipeline satisfies. The selector registers this notebook in `METADATA_NOTEBOOK_REGISTRY` using the metadata target configured by `00_env_config`.

In [ ]:
PIPELINE_STARTED_AT = datetime.now(timezone.utc).replace(microsecond=0).isoformat()
RUN_ID = RUN_CONTEXT.run_id
ENV_NAME = ENV
PIPELINE_NAME = "sample_agreement_pipeline"

widget_select_agreement(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    notebook_type="02_pipeline",
    pipeline_name=PIPELINE_NAME,
)

AGREEMENT = get_selected_agreement()
AGREEMENT_ID = AGREEMENT.get("agreement_id", "")
AGREEMENT_CONTRACT_VERSION = AGREEMENT.get("agreement_contract_version", AGREEMENT.get("contract_version", ""))
NOTEBOOK_REGISTRY_ID = AGREEMENT.get("notebook_registry_id", AGREEMENT.get("registration_id", ""))
NOTEBOOK_ID = AGREEMENT.get("notebook_id", RUN_CONTEXT.runtime_metadata.get("currentNotebookId", ""))


## 4. Define and read many source datasets

Edit this section for your sources. Add one entry per source table or file. Each source owns its schema, drift, and DQ presets.

In [ ]:
USE_SAMPLE_DATA = True
DATASET_NAME = "sample_agreement_dataset" if USE_SAMPLE_DATA else "CHANGE_ME_dataset"

SOURCE_DEFINITIONS = {
    "minimal_source": {
        "dataset_name": DATASET_NAME,
        "table_name": "minimal_source" if USE_SAMPLE_DATA else "CHANGE_ME_source_table",
        "kind": "csv",  # lakehouse | warehouse | csv | parquet | excel
        "layer": "source",
        "path": "Files/sample/minimal_source.csv" if USE_SAMPLE_DATA else "Files/CHANGE_ME/source_file.csv",
        "stage": "source",
        "schema_preset": "allow_new_columns",
        "drift_preset": "changing_data",
        "dq_preset": "approved_rules",
        "expected_schema": {
            "customer_id": "bigint",
            "event_ts": "string",
            "status": "string",
            "amount": "double",
            "email": "string",
            "country_code": "string",
        },
        "distribution_columns": ["status", "amount", "country_code"],
    },
    # Add more sources by copying the shape above.
    # "another_source": {
    #     "dataset_name": DATASET_NAME,
    #     "table_name": "CHANGE_ME_another_source",
    #     "kind": "lakehouse",
    #     "layer": "source",
    #     "stage": "source",
    #     "schema_preset": "strict",
    #     "drift_preset": "monitor_changing_data",
    #     "dq_preset": "skip",
    #     "expected_schema": {"id": "bigint"},
    # },
}

source_dfs = read_pipeline_sources(SOURCE_DEFINITIONS, config=CONFIG, env=ENV_NAME, spark_session=spark)

# Friendly aliases for beginners editing the transformation section.
df_minimal_source = source_dfs["minimal_source"]


## 5. Profile each source DataFrame

Profiles are reused for drift baselines and catalogue evidence.

In [ ]:
source_profiles = profile_pipeline_datasets(source_dfs, SOURCE_DEFINITIONS)


## 6. Check each source schema

Each source uses its own `schema_preset`. Blocking failures stop before transformation.

In [ ]:
source_schema_results = run_schema_guardrails(source_dfs, SOURCE_DEFINITIONS)
for result in source_schema_results.values():
    stop_if_failed(result)


## 7. Check each source for data drift

Each source uses its own `drift_preset` and its own catalogue baseline.

In [ ]:
source_drift_results = run_data_drift_guardrails(
    source_dfs,
    SOURCE_DEFINITIONS,
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
)
for result in source_drift_results.values():
    stop_if_failed(result)


## 8. Check each source with DQ guardrails

Each source uses its own `dq_preset`. Warning severity writes full data; error severity stops before downstream writes. No row filtering in v1.

In [ ]:
source_dq_results = run_dq_guardrails(source_dfs, SOURCE_DEFINITIONS, config=CONFIG, env=ENV_NAME, spark_session=spark)
for result in source_dq_results.values():
    print(result)
    stop_if_failed(result)


## 9. Write source catalogue evidence

FabricOps enriches profile rows with agreement, notebook registry, guardrail, and DQ result columns before writing `METADATA_DATA_CATALOGUE`.

In [ ]:
source_catalogue_status = write_catalogue_evidence(
    source_profiles,
    SOURCE_DEFINITIONS,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    schema_results=source_schema_results,
    drift_results=source_drift_results,
    dq_results=source_dq_results,
)


## 10. User-defined transformation section

This is the main DIY section. Keep business logic here and reusable governance plumbing in package helpers.

In [ ]:
df_output = (
    df_minimal_source
    .withColumn(
        "amount_band",
        F.when(F.col("amount") >= F.lit(100), F.lit("high"))
        .when(F.col("amount") >= F.lit(25), F.lit("medium"))
        .otherwise(F.lit("low")),
    )
)

# Add more transformations or joins here. For many sources, use the friendly
# aliases above or access source_dfs["source_alias"].


## 11. Define target DataFrames and add audit columns

Edit this section for your outputs. Add one entry per target. Each target owns its schema, drift, and DQ presets.

In [ ]:
TARGET_DEFINITIONS = {
    "sample_output": {
        "dataset_name": DATASET_NAME,
        "table_name": "sample_agreement_output" if USE_SAMPLE_DATA else "CHANGE_ME_target_table",
        "kind": "lakehouse",  # lakehouse | warehouse
        "layer": "product",
        "stage": "target",
        "mode": "overwrite",
        "schema_preset": "strict",
        "drift_preset": "changing_data",
        "dq_preset": "approved_rules",
        "expected_schema": {
            "customer_id": "bigint",
            "event_ts": "string",
            "status": "string",
            "amount": "double",
            "email": "string",
            "country_code": "string",
            "amount_band": "string",
            "_fabricops_run_id": "string",
            "_fabricops_pipeline_name": "string",
            "_fabricops_created_at": "string",
        },
        "distribution_columns": ["status", "amount", "amount_band", "country_code"],
    },
    # Add more targets by copying the shape above and mapping an alias to a DataFrame below.
}

target_dfs = {
    "sample_output": df_output,
}

target_dfs = add_runtime_audit_columns(target_dfs, run_id=RUN_ID, pipeline_name=PIPELINE_NAME)
df_output = target_dfs["sample_output"]


## 12. Check each target schema

Target checks mirror source checks and run before publication.

In [ ]:
target_schema_results = run_schema_guardrails(target_dfs, TARGET_DEFINITIONS)
for result in target_schema_results.values():
    stop_if_failed(result)


## 13. Check each target for data drift

Target drift compares proposed target DataFrames before any target write occurs.

In [ ]:
target_drift_results = run_data_drift_guardrails(
    target_dfs,
    TARGET_DEFINITIONS,
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
)
for result in target_drift_results.values():
    stop_if_failed(result)


## 14. Check each target with DQ guardrails

Approved active DQ rules are evaluated as aggregate guardrails before writing full target datasets.

In [ ]:
# Warning severity writes full data; error severity stops before write. No row filtering in v1.
target_dq_results = run_dq_guardrails(target_dfs, TARGET_DEFINITIONS, config=CONFIG, env=ENV_NAME, spark_session=spark)
for target_name, result in target_dq_results.items():
    print(result)
    stop_if_failed(result)
    if "dataframe" in result:
        target_dfs[target_name] = result["dataframe"]

df_output = target_dfs["sample_output"]


## 15. Write target catalogue evidence

FabricOps writes target profile evidence with DQ result columns using the same shape as source evidence.

In [ ]:
target_profiles = profile_pipeline_datasets(target_dfs, TARGET_DEFINITIONS)

target_catalogue_status = write_catalogue_evidence(
    target_profiles,
    TARGET_DEFINITIONS,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    schema_results=target_schema_results,
    drift_results=target_drift_results,
    dq_results=target_dq_results,
)


## 16. Write target tables

Only target definitions and DataFrames are passed to the writer. Advanced partitioning options belong in `TARGET_DEFINITIONS`.

In [ ]:
target_write_status = write_pipeline_targets(target_dfs, TARGET_DEFINITIONS, config=CONFIG, env=ENV_NAME)


## 17. Capture many-to-many lineage

Define source-to-target relationships at the business level. FabricOps builds and writes metadata rows tied to the selected notebook registration.

In [ ]:
LINEAGE_RELATIONSHIPS = [
    {
        "sources": ["minimal_source"],
        "targets": ["sample_output"],
        "operation": "derive amount band and publish governed target",
        "description": "Sample source rows are transformed into the sample agreement output.",
    },
]

lineage_result = write_pipeline_lineage(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    source_definitions=SOURCE_DEFINITIONS,
    target_definitions=TARGET_DEFINITIONS,
    relationships=LINEAGE_RELATIONSHIPS,
    dataset_name=DATASET_NAME,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
)


## 18. Write runtime summary

Runtime evidence is stored in `METADATA_PIPELINE_RUNS` and displayed for operational support.

In [ ]:
catalogue_status = "written" if source_catalogue_status and target_catalogue_status else "not_written"

run_summary = write_pipeline_run_summary(
    spark=spark,
    config=CONFIG,
    env=ENV_NAME,
    run_id=RUN_ID,
    agreement_id=AGREEMENT_ID,
    agreement_contract_version=AGREEMENT_CONTRACT_VERSION,
    notebook_registry_id=NOTEBOOK_REGISTRY_ID,
    notebook_id=NOTEBOOK_ID,
    pipeline_name=PIPELINE_NAME,
    started_at=PIPELINE_STARTED_AT,
    completed_at=datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    status="completed",
    source_definitions=SOURCE_DEFINITIONS,
    target_definitions=TARGET_DEFINITIONS,
    source_schema_results=source_schema_results,
    target_schema_results=target_schema_results,
    source_drift_results=source_drift_results,
    target_drift_results=target_drift_results,
    source_dq_results=source_dq_results,
    target_dq_results=target_dq_results,
    lineage_status=lineage_result.get("status", "unknown"),
    catalogue_status=catalogue_status,
    message="Pipeline completed and metadata evidence was written.",
)

display(run_summary)
